# SegFormer MiT-B5 — WL Bruise Segmentation (ORC / SLURM Jupyter Lab)

Trains a **SegFormer-B5** baseline with the *same custom loop / recipe* as the core SegFormer models (B2 teacher, B0 direct) in `bruise_colab_final.ipynb` — Dice+BCE, AdamW backbone/head LR split (6e-5 / 6e-4), warmup→poly schedule, AMP, VRAM-probed batch → effective 8, epoch selection on **val mean Dice @ 0.50**, threshold swept on **VAL**, scored **once** on the 185-image consensus **TEST** set at matched 640 geometry (`train_segformer_b5_baseline.py`: *train → sweep → score*).

### ORC session assumptions
- Launched via OnDemand Jupyter Lab with `--notebook-dir=/scratch/$USER`, **GPU** partition, GPU type **3g.40gb** (a 40 GB MIG slice ≈ 3/7 of an A100), 1 GPU, 8 cores, 12 h wall-time.
- Everything lives on **`/scratch/$USER`** — it persists between sessions, so checkpoints survive a 12 h wall-time kill and you can **resume** (no Google Drive needed).

### Before you run
1. Copy **`segformer_b5_bruise_pipeline.zip`** (~2.9 GB: code + data + the bundled `nvidia/mit-b5` ImageNet encoder) to `/scratch/$USER/` (e.g. `scp` / Globus / `cp` from your home dir).
2. Make sure this Jupyter kernel has **PyTorch with GPU** (cell 2 checks; cell 4 installs `transformers`/`albumentations` if missing). If `torch.cuda.is_available()` is False, switch to a conda kernel that has GPU torch.
3. **No internet needed for weights** — the MiT-B5 encoder is inside the zip and loaded with `HF_HUB_OFFLINE=1`.

### Epoch budget vs the 12 h wall-time
- B5 is the **largest** MiT encoder (~85 M params); on a 3g.40gb MIG slice at 640×640 expect a *few minutes per epoch*. **Measure first** with the 1-epoch smoke test (set `EPOCHS = 1`), read the per-epoch seconds from the log, then size the run.
- Default is the reference **100 epochs / patience 15** — early stopping usually ends the run well before 100. If the 12 h wall-time kills it mid-run, just relaunch and re-run the cells — the script **auto-resumes** from `resume_checkpoint.pt` on `/scratch` (written every epoch; at most 1 epoch lost).
- If the VRAM probe lands on `micro_batch=1`, set `GRAD_CHECKPOINT = True` and restart the run — checkpointing trades ~30% speed for a larger micro-batch.

## 1 · GPU / SLURM / env check

In [ ]:
import os
!nvidia-smi
import torch
print('\npython env :', os.sys.executable)
print('torch      :', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
print('SLURM job  :', os.environ.get('SLURM_JOB_ID','?'),
      '| cpus', os.environ.get('SLURM_CPUS_PER_TASK', os.environ.get('SLURM_CPUS_ON_NODE','?')),
      '| gpu', os.environ.get('SLURM_JOB_GPUS', os.environ.get('CUDA_VISIBLE_DEVICES','?')))
assert torch.cuda.is_available(), 'GPU not visible to torch — switch to a CUDA-enabled kernel/conda env.'

## 2 · Config — edit these
Paths default to `/scratch/$USER`. `EPOCHS = 1` is the smoke test; `100` is the reference recipe (early stopping at patience 15 usually ends it sooner). Resume is **automatic** — nothing to toggle.

In [ ]:
import os
USER = os.environ.get('USER', os.environ.get('LOGNAME', 'user'))
SCRATCH = f'/scratch/{USER}'

# ===== EDIT ME =====
ZIP_PATH   = f'{SCRATCH}/segformer_b5_bruise_pipeline.zip'   # where you copied the package
WORK       = f'{SCRATCH}/segformer_b5_bruise_pipeline'       # unzip target (persistent /scratch)

EPOCHS          = 100     # 1 = smoke test. 100 = reference recipe (patience 15).
SEEDS           = [42]    # e.g. [0, 1, 2] for the multi-seed run
GRAD_CHECKPOINT = False   # True if the probe lands on micro_batch=1 (B5 is big)
EVAL_ONLY       = False   # True to re-run only the val sweep + test scoring
print('USER =', USER, '| WORK =', WORK, '| EPOCHS =', EPOCHS, '| SEEDS =', SEEDS)

## 3 · Unzip the package (on `/scratch`, persistent)

In [ ]:
import os, zipfile, time
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH} — copy segformer_b5_bruise_pipeline.zip to {SCRATCH}/ first.'
if not os.path.isdir(WORK):
    t = time.time()
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(SCRATCH)   # archive holds a top-level segformer_b5_bruise_pipeline/ folder
    print(f'unzipped in {time.time()-t:.0f}s')
else:
    print('already unzipped:', WORK)
assert os.path.isdir(f'{WORK}/data/train/images'), 'data/ missing — is this the full ~2.9 GB zip?'
assert os.path.exists(f'{WORK}/pretrained_weights/segformer_mit_b5/pytorch_model.bin'), \
    'pretrained_weights/segformer_mit_b5 missing — the MiT-B5 encoder must be inside the zip (offline load).'
assert os.path.exists(f'{WORK}/ita_labels/wl_test_per_image_ita.csv'), \
    'ita_labels/ missing — old zip without the fairness labels; re-copy the current zip.'
# guard: ensure the script is the canonical-split version
_sig = open(f'{WORK}/train_segformer_b5_baseline.py').read()
assert 'resolve_train_val' in _sig, 'Stale zip: train_segformer_b5_baseline.py lacks the canonical-split logic.'
!ls {WORK}

## 4 · Ensure transformers / albumentations are installed
If the kernel already has them, this is a no-op. Otherwise it tries `pip install --user` (works if the ORC compute node has internet/a proxy). If that fails, create a conda env with `torch`+`transformers`+`albumentations` on the **login node** first and relaunch Jupyter against it. (The **model weights** never need internet — they're in the zip.)

In [ ]:
import importlib, subprocess, sys
def _have(m):
    try: importlib.import_module(m); return True
    except Exception: return False
missing = [m for m in ('transformers', 'albumentations', 'cv2', 'pandas', 'tqdm', 'scipy') if not _have(m)]
pipname = {'cv2': 'opencv-python-headless'}
if not missing:
    print('all deps already present — nothing to install.')
else:
    print('missing:', missing, '— attempting pip install --user ...')
    rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--user', '-q',
                          *[pipname.get(m, m) for m in missing]])
    still = [m for m in missing if not _have(m)]
    if rc != 0 or still:
        raise SystemExit(f'Install failed for {still} (likely no internet on the compute node). '
                         'Pre-build a conda env on the login node and use that kernel.')
    print('installed into ~/.local')
import transformers, albumentations, cv2
print('transformers', transformers.__version__, '| albumentations', albumentations.__version__,
      '| cv2', cv2.__version__)

## 5 · Env fixes for ORC
- **Offline HF**: `HF_HUB_OFFLINE` / `TRANSFORMERS_OFFLINE` so the bundled MiT-B5 loads from disk without a hub lookup (the script also sets these, belt-and-braces).
- **LD_LIBRARY_PATH**: strip system cuda/cudnn dirs — the pip PyTorch wheel bundles its own cuDNN, and a mismatched system cuDNN on the path otherwise crashes training.
- **Workers**: dataloader workers pinned to the **allocated** cores so the MIG slice doesn't over-subscribe the node's full CPU list.

In [ ]:
import os, torch

# (a) offline HuggingFace — weights load from the zip, never the hub
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# (b) let PyTorch use its bundled cuDNN (drop system cuda/cudnn from LD_LIBRARY_PATH)
kept = [p for p in os.environ.get('LD_LIBRARY_PATH','').split(':')
        if p and 'cuda' not in p.lower() and 'cudnn' not in p.lower()]
os.environ['LD_LIBRARY_PATH'] = ':'.join(kept)
print('cuDNN:', torch.backends.cudnn.version(), '| gpu:', torch.cuda.is_available())

# (c) dataloader workers pinned to the SLURM allocation
ncpu = int(os.environ.get('SLURM_CPUS_PER_TASK', os.environ.get('SLURM_CPUS_ON_NODE', '8')))
NWORKERS = max(2, ncpu - 1)
print(f'cores={ncpu} -> dataloader workers={NWORKERS}')

## 6 · Verify the data + weights (697/134 split, files present)

In [ ]:
%cd {WORK}
!python check_package.py

## 7 · Train → sweep threshold on VAL → score on TEST
Live epoch logs stream below (and to `results_segformer_b5/runs/.../training_history.csv`). If the 12 h wall-time kills the job mid-run, just relaunch, re-run cells 1–6, then re-run this cell — the script **auto-resumes** from the last `/scratch` checkpoint (probe skipped, optimizer/schedule restored). After training it sweeps thresholds 0.05–0.95 on val and evaluates the test set once at the best one.

In [ ]:
flags = []
if GRAD_CHECKPOINT: flags.append('--grad-checkpoint')
if EVAL_ONLY:       flags.append('--eval-only')
cmd = f'''python -u train_segformer_b5_baseline.py \
  --train-manifest manifests/train_manifest.csv \
  --test-manifest  manifests/test_manifest.csv \
  --data-root "{WORK}" --out-dir "{WORK}/results_segformer_b5" \
  --pretrained "{WORK}/pretrained_weights/segformer_mit_b5" \
  --epochs {EPOCHS} --workers {NWORKERS} \
  --seeds {' '.join(str(s) for s in SEEDS)} {' '.join(flags)}'''
print(cmd, '\n' + '='*70)
get_ipython().system(cmd)

## 8 · Results
Everything is already on persistent `/scratch` — nothing to copy back.

In [ ]:
import json, pandas as pd
res = f'{WORK}/results_segformer_b5/results/segformer_b5_FINAL.json'
summ = json.load(open(res))
print(json.dumps(summ, indent=2))
print('\n  mean Dice   %.4f' % summ['mean_dice'])
print('  median Dice %.4f'   % summ['median_dice'])
print('  mean IoU    %.4f'   % summ['mean_iou'])
print('  miss rate   %.2f%%  (%d/%d)' % (summ['complete_miss_rate']*100,
                                          summ['complete_miss_count'], summ['n_images']))
print('\n  full outputs :', f'{WORK}/results_segformer_b5/results')
print('  run dirs     :', f'{WORK}/results_segformer_b5/runs')
hist = f'{WORK}/results_segformer_b5/runs/segformer_b5__seed{SEEDS[0]}/training_history.csv'
print('\nlast epochs:')
print(pd.read_csv(hist).tail(5).to_string(index=False))

## 9 · Fairness across skin tone (ITA) — all 185 test images
Per-image Dice (averaged over seeds if you ran several), stratified by the 5 ITA skin-tone groups, via the **same `fairness_analysis`** used for the 5 core models and the U-Net/DeepLab baselines: per-group median Dice + IQR + bootstrap 95% CI, mean recall, complete-miss rate, Kruskal–Wallis across groups, Bonferroni-corrected pairwise Mann–Whitney. **Exploratory at n=28 subjects** — each ITA group has only ~9–17 subjects, so read direction, not significance.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats as _st

# --- per-image test results, averaged over seeds (same as baselines notebook §10) ----
frames = []
for s in SEEDS:
    f = f'{WORK}/results_segformer_b5/runs/segformer_b5__seed{s}/test_per_image.csv'
    if os.path.exists(f):
        frames.append(pd.read_csv(f))
assert frames, 'no test_per_image.csv found — run cell 7 first.'
per_image = (pd.concat(frames).groupby('stem', as_index=False)
             .agg({'dice': 'mean', 'recall': 'mean',
                   'pred_positive_pixels': 'mean', 'gt_positive_pixels': 'first'}))
assert len(per_image) == 185, f'expected 185 test images, got {len(per_image)}'

# --- ITA labels (bundled in the zip; keyed by stem) ----------------------------------
ITA = pd.read_csv(f'{WORK}/ita_labels/wl_test_per_image_ita.csv')[
    ['stem', 'skin_tone_category', 'ita_group_index_5']]

# --- fairness_analysis (ported verbatim from the core-model analysis notebooks) ------
def _bootstrap_ci(values, n=2000, seed=0):
    if len(values) < 2: return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    meds = [np.median(rng.choice(values, size=len(values), replace=True)) for _ in range(n)]
    return float(np.percentile(meds, 2.5)), float(np.percentile(meds, 97.5))

def fairness_analysis(per_image_df, ita, model_name):
    df = per_image_df.merge(ita, on='stem', how='left', validate='one_to_one')
    assert df['ita_group_index_5'].notna().all(), 'stems missing ITA labels'
    per_group, samples = [], []
    for gidx, g in sorted(df.groupby('ita_group_index_5'), key=lambda kv: kv[0]):
        vals = g['dice'].to_numpy(); lo, hi = _bootstrap_ci(vals)
        per_group.append({'model': model_name, 'ita_group_index_5': int(gidx),
                          'skin_tone_category': g['skin_tone_category'].iloc[0], 'n_images': len(g),
                          'median_dice': float(np.median(vals)),
                          'iqr_dice': float(np.percentile(vals, 75) - np.percentile(vals, 25)),
                          'ci95_lo': lo, 'ci95_hi': hi, 'mean_recall': float(g['recall'].mean()),
                          'miss_rate': float(((g['pred_positive_pixels'] == 0) & (g['gt_positive_pixels'] > 0)).mean())})
        samples.append(vals)
    H, p = _st.kruskal(*samples)
    pairs = [(i, j) for i in range(len(samples)) for j in range(i + 1, len(samples))]
    pairwise = []
    for i, j in pairs:
        pv = _st.mannwhitneyu(samples[i], samples[j], alternative='two-sided').pvalue
        adj = min(1.0, pv * len(pairs))
        pairwise.append({'model': model_name, 'group_a': per_group[i]['skin_tone_category'],
                         'group_b': per_group[j]['skin_tone_category'], 'pvalue': pv,
                         'bonferroni_p': adj, 'significant': bool(adj < 0.05)})
    pg = pd.DataFrame(per_group)
    best, worst = pg.loc[pg['median_dice'].idxmax()], pg.loc[pg['median_dice'].idxmin()]
    stat = {'model': model_name, 'kruskal_H': float(H), 'kruskal_p': float(p), 'significant': bool(p < 0.05),
            'fairness_gap': float(best['median_dice'] - worst['median_dice']),
            'best_group': best['skin_tone_category'], 'worst_group': worst['skin_tone_category'],
            'max_miss_rate_gap': float(pg['miss_rate'].max() - pg['miss_rate'].min())}
    return {'per_group': pg, 'pairwise': pd.DataFrame(pairwise), 'stats': stat}

out = fairness_analysis(per_image, ITA, 'segformer_b5')
RES = f'{WORK}/results_segformer_b5/results'
out['per_group'].to_csv(f'{RES}/segformer_b5_fairness_per_group.csv', index=False)
out['pairwise'].to_csv(f'{RES}/segformer_b5_fairness_pairwise.csv', index=False)
pd.DataFrame([out['stats']]).to_csv(f'{RES}/segformer_b5_fairness_stats.csv', index=False)

print(out['per_group'][['skin_tone_category', 'n_images', 'median_dice', 'iqr_dice',
                        'ci95_lo', 'ci95_hi', 'mean_recall', 'miss_rate']].to_string(index=False))
s = out['stats']
print(f"\nKruskal-Wallis H={s['kruskal_H']:.2f} p={s['kruskal_p']:.4f} "
      f"(significant={s['significant']})")
print(f"fairness gap (median Dice): {s['fairness_gap']:.4f}  "
      f"best={s['best_group']}  worst={s['worst_group']}")
print(f"max miss-rate gap: {s['max_miss_rate_gap']*100:.2f}%")
sig = out['pairwise'][out['pairwise'].significant]
print('\npairwise (Bonferroni) significant:', 'none' if sig.empty else '')
if not sig.empty: print(sig.to_string(index=False))

try:
    import matplotlib.pyplot as plt
    GROUP_ORDER = ['Light (II-III)', 'Intermediate (III-IV)', 'Tan (IV)', 'Brown (V)', 'Dark (VI)']
    pg = out['per_group'].set_index('skin_tone_category').reindex(
        [g for g in GROUP_ORDER if g in set(out['per_group'].skin_tone_category)])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(pg.index, pg['median_dice'],
           yerr=[pg['median_dice'] - pg['ci95_lo'], pg['ci95_hi'] - pg['median_dice']],
           capsize=4, color='#4477aa')
    ax.set_ylabel('median Dice'); ax.set_ylim(0, 1); ax.grid(axis='y', alpha=0.3)
    ax.set_title('SegFormer-B5 median Dice by ITA group (bootstrap 95% CI, exploratory n=28)')
    plt.xticks(rotation=15); plt.tight_layout()
    plt.savefig(f'{RES}/segformer_b5_fairness_by_group.png', dpi=140, bbox_inches='tight')
    plt.show()
except ImportError:
    print('(matplotlib not installed — table only)')